# 01 — Coleta e Geração de Dados Fictícios

Este notebook demonstra como gerar um dataset fictício de **churn de clientes**
para servir de exemplo no pipeline de Ciência de Dados.

**Tema:** Previsão de Churn em Telecomunicações  
**Objetivo:** Criar uma base com ~1000 registros e 12 variáveis (numéricas e categóricas).

---

## 1.1 Importações e Configurações

In [ ]:
import os
import numpy as np
import pandas as pd

# Reprodutibilidade
SEED = 42
np.random.seed(SEED)

N_CLIENTES = 1000
CAMINHO_SAIDA = os.path.join("..", "dados", "bruto", "dataset_clientes.csv")

## 1.2 Geração do Dataset

O dataset simula dados de uma empresa de telecomunicações com as seguintes variáveis:

| Variável | Tipo | Descrição |
|----------|------|-----------|
| `id_cliente` | int | Identificador único |
| `idade` | int | Idade do cliente (18-70) |
| `sexo` | cat | M/F |
| `renda_mensal` | float | Renda mensal (log-normal) |
| `tempo_conta_meses` | int | Tempo de conta (1-120) |
| `num_produtos` | int | Produtos contratados (1-4) |
| `tem_credito` | bin | Possui crédito (0/1) |
| `saldo` | float | Saldo da conta (exponencial) |
| `ativo` | bin | Cliente ativo (0/1) |
| `num_reclamacoes` | int | Reclamações (Poisson) |
| `satisfacao` | ord | Satisfação (1-5) |
| `canal_aquisicao` | cat | Canal de aquisição |
| `churn` | bin | **Target**: cancelou? (0/1) |

In [ ]:
dados = {
    "id_cliente": range(1, N_CLIENTES + 1),
    "idade": np.random.randint(18, 70, size=N_CLIENTES),
    "sexo": np.random.choice(["M", "F"], size=N_CLIENTES, p=[0.48, 0.52]),
    "renda_mensal": np.round(np.random.lognormal(mean=8.5, sigma=0.6, size=N_CLIENTES), 2),
    "tempo_conta_meses": np.random.randint(1, 120, size=N_CLIENTES),
    "num_produtos": np.random.choice([1, 2, 3, 4], size=N_CLIENTES, p=[0.45, 0.35, 0.15, 0.05]),
    "tem_credito": np.random.choice([0, 1], size=N_CLIENTES, p=[0.3, 0.7]),
    "saldo": np.round(np.random.exponential(scale=50000, size=N_CLIENTES), 2),
    "ativo": np.random.choice([0, 1], size=N_CLIENTES, p=[0.15, 0.85]),
    "num_reclamacoes": np.random.poisson(lam=1.2, size=N_CLIENTES),
    "satisfacao": np.random.choice([1, 2, 3, 4, 5], size=N_CLIENTES, p=[0.05, 0.10, 0.30, 0.35, 0.20]),
    "canal_aquisicao": np.random.choice(
        ["Loja Física", "Online", "Telefone", "Indicação"],
        size=N_CLIENTES, p=[0.30, 0.40, 0.15, 0.15]
    ),
}

df = pd.DataFrame(dados)

## 1.3 Variável-alvo (churn) com lógica realista

A probabilidade de churn é modelada como função de:
- Número de reclamações (mais reclamações → maior churn)
- Satisfação (menor satisfação → maior churn)
- Tempo de conta (menos tempo → maior churn)
- Status ativo (inativo → maior churn)

In [ ]:
prob_churn = (
    0.05
    + 0.08 * (df["num_reclamacoes"] / df["num_reclamacoes"].max())
    + 0.15 * (1 - df["satisfacao"] / 5)
    + 0.10 * (1 - df["tempo_conta_meses"] / 120)
    + 0.05 * (1 - df["ativo"])
)
prob_churn = prob_churn.clip(0.02, 0.95)
df["churn"] = np.random.binomial(1, prob_churn)

# Introduzir valores ausentes realistas (~5%)
idx_renda = np.random.choice(N_CLIENTES, size=int(N_CLIENTES * 0.05), replace=False)
df.loc[idx_renda, "renda_mensal"] = np.nan

idx_satisfacao = np.random.choice(N_CLIENTES, size=int(N_CLIENTES * 0.03), replace=False)
df.loc[idx_satisfacao, "satisfacao"] = np.nan

print(f"Dataset gerado: {df.shape}")
print(f"\nDistribuição de churn:\n{df['churn'].value_counts()}")
print(f"\nValores ausentes:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
df.head()

## 1.4 Salvar Dataset

In [ ]:
os.makedirs(os.path.dirname(CAMINHO_SAIDA), exist_ok=True)
df.to_csv(CAMINHO_SAIDA, index=False, encoding="utf-8")
print(f"✓ Dataset salvo em: {CAMINHO_SAIDA}")